### Learning Specific data science libraries like Pandas, Scikit-learn, numpy & others for data cleaning and manipulation.
#### Covering below Scenarios:
#####	1. Combining and Cleaning data: ML data often comes from several sources and must be combined before use.		 
#####	2. Missing-data Imputation (assigning a value to something by inference from other products): Real-world data may contain missing values because of human errors, data-entry problems, transmission failures or faulty recordings.		
#####	3. Categorical-data Encoding: ML models generally require numerical input, so categorical values must be encoded.			
#####	4. Centering: Used to put numerical features onto a comparable scale.

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

In [2]:
## Raw Data
# Source A: Core HR profile logs
hr_data = {
    'Emp_ID':[101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eva', 'Frank', 'Grace', 'Henry', 'Ivanka', 'Jack'],
    'Dept': ['Tech', 'HR', 'tech', 'Sales', 'Finance', 'HR', 'Tech', 'Sales', 'Finance', 'Tech'],        # 'tech' mismatch
    'Seniority_Level': ['Mid', 'Junior', 'Senior', 'Junior', 'Senior', 'Mid', 'Junior', 'Senior', 'Mid', 'Executive']
}
df_hr = pd.DataFrame(hr_data)

print("--- Raw Dataset - 1.HR Data ---")
print(df_hr.to_string(index=False))
print("\n" + "="*70 + "\n")

# Source B: Payroll and metric systems (Notice order mismatches and a duplicate row)
payroll_data = {
    'Emp_ID': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 110],                                         # 110 is duplicated
    'Salary_USD': [105000, 65000, 52000, 72000, np.nan, 58000, 61000, np.nan, 88000, 210000, 210000], 
    'Rating': [4.5, 3.8, np.nan, 4.0, 4.2, 3.5, np.nan, 4.1, 4.8, 5.0, 5.0]
}
df_payroll = pd.DataFrame(payroll_data)

print("--- Raw Dataset - 2.Payroll Data ---")
print(df_payroll.to_string(index=False))
print("\n" + "="*70 + "\n")

--- Raw Dataset - 1.HR Data ---
 Emp_ID    Name    Dept Seniority_Level
    101   Alice    Tech             Mid
    102     Bob      HR          Junior
    103 Charlie    tech          Senior
    104   David   Sales          Junior
    105     Eva Finance          Senior
    106   Frank      HR             Mid
    107   Grace    Tech          Junior
    108   Henry   Sales          Senior
    109  Ivanka Finance             Mid
    110    Jack    Tech       Executive


--- Raw Dataset - 2.Payroll Data ---
 Emp_ID  Salary_USD  Rating
    101    105000.0     4.5
    102     65000.0     3.8
    103     52000.0     NaN
    104     72000.0     4.0
    105         NaN     4.2
    106     58000.0     3.5
    107     61000.0     NaN
    108         NaN     4.1
    109     88000.0     4.8
    110    210000.0     5.0
    110    210000.0     5.0




In [3]:
# =====================================================================
# SCENARIO 1: COMBINING AND CLEANING DATA
# =====================================================================
# A. Removing Duplicates from Payroll source
df_payroll = df_payroll.drop_duplicates()

# B. Joining data sets by columns (Merging on unique Emp_ID key)
df = pd.merge(df_hr, df_payroll, on='Emp_ID', how='inner')

# C. Standardizing formats, units & naming conventions
df['Dept'] = df['Dept'].str.title() # Fixes lowercase 'tech' to 'Tech'
df = df.rename(columns={'Dept': 'Department'})

# D. Appending, deleting, and filtering rows or columns
df['Active_Status'] = 'Active'         # Appending a column
df = df.drop(columns=['Active_Status']) # Deleting a column
df = df[df['Salary_USD'] != 52000]      # Filtering out rows (e.g., removing Charlie)

print("--- SCENARIO 1: COMBINED & CLEANED DATAFRAME ---")
print(df.to_string(index=False))
print("\n" + "="*70 + "\n")

--- SCENARIO 1: COMBINED & CLEANED DATAFRAME ---
 Emp_ID   Name Department Seniority_Level  Salary_USD  Rating
    101  Alice       Tech             Mid    105000.0     4.5
    102    Bob         Hr          Junior     65000.0     3.8
    104  David      Sales          Junior     72000.0     4.0
    105    Eva    Finance          Senior         NaN     4.2
    106  Frank         Hr             Mid     58000.0     3.5
    107  Grace       Tech          Junior     61000.0     NaN
    108  Henry      Sales          Senior         NaN     4.1
    109 Ivanka    Finance             Mid     88000.0     4.8
    110   Jack       Tech       Executive    210000.0     5.0




In [4]:
# =====================================================================
# SCENARIO 2: MISSING-DATA IMPUTATION
# =====================================================================
# A. Numerical Imputation using Mean/Median (Scikit-Learn SimpleImputer)
# We choose Median for Salary because it is robust against executive outliers (like Jack's $210k)

num_imputer = SimpleImputer(strategy='median')
df['Salary_USD'] = num_imputer.fit_transform(df[['Salary_USD']])

# B. Numerical Imputation using Mean for Rating

rating_imputer = SimpleImputer(strategy='mean')
df['Rating'] = rating_imputer.fit_transform(df[['Rating']])

print("--- SCENARIO 2: DATA AFTER IMPUTING NULLS ---")
print(df[['Name', 'Salary_USD', 'Rating']].to_string(index=False))
print("\n" + "="*70 + "\n")

--- SCENARIO 2: DATA AFTER IMPUTING NULLS ---
  Name  Salary_USD  Rating
 Alice    105000.0  4.5000
   Bob     65000.0  3.8000
 David     72000.0  4.0000
   Eva     72000.0  4.2000
 Frank     58000.0  3.5000
 Grace     61000.0  4.2375
 Henry     72000.0  4.1000
Ivanka     88000.0  4.8000
  Jack    210000.0  5.0000




In [5]:
# =====================================================================
# SCENARIO 3: CATEGORICAL-DATA ENCODING
# =====================================================================
# A. Ordinal Encoding (Hierarchy exists: Junior -> Mid -> Senior -> Executive)

level_order = [['Junior', 'Mid', 'Senior', 'Executive']]
ord_encoder = OrdinalEncoder(categories=level_order)
df['Seniority_Encoded'] = ord_encoder.fit_transform(df[['Seniority_Level']])

# B. Nominal One-Hot Encoding (No hierarchy exists for Departments)

ohe_encoder = OneHotEncoder(sparse_output=False)
encoded_depts = ohe_encoder.fit_transform(df[['Department']])
dept_column_names = ohe_encoder.get_feature_names_out(['Department'])

# Map the array back into our primary dataframe safely

df[dept_column_names] = encoded_depts

print("--- SCENARIO 3: CATEGORICAL ENCODING PREVIEW ---")
print(df[['Name', 'Seniority_Level', 'Seniority_Encoded'] + list(dept_column_names)].to_string(index=False))
print("\n" + "="*70 + "\n")

--- SCENARIO 3: CATEGORICAL ENCODING PREVIEW ---
  Name Seniority_Level  Seniority_Encoded  Department_Finance  Department_Hr  Department_Sales  Department_Tech
 Alice             Mid                1.0                 0.0            0.0               0.0              1.0
   Bob          Junior                0.0                 0.0            1.0               0.0              0.0
 David          Junior                0.0                 0.0            0.0               1.0              0.0
   Eva          Senior                2.0                 1.0            0.0               0.0              0.0
 Frank             Mid                1.0                 0.0            1.0               0.0              0.0
 Grace          Junior                0.0                 0.0            0.0               0.0              1.0
 Henry          Senior                2.0                 0.0            0.0               1.0              0.0
Ivanka             Mid                1.0              

In [ ]:
# =====================================================================
# SCENARIO 4: CENTERING & STANDARDIZATION
# =====================================================================
# Manual Demo of Centering (x' = x - mean)

salary_mean = df['Salary_USD'].mean()
df['Centered_Salary_Manual'] = df['Salary_USD'] - salary_mean

# Industrial Production Way: Using Scikit-Learn StandardScaler
# This centers the data (Mean = 0) AND scales it to unit variance (Standard Deviation = 1)

scaler = StandardScaler()
df['Standardized_Salary_Sklearn'] = scaler.fit_transform(df[['Salary_USD']])

print("--- SCENARIO 4: CENTERING & STANDARDIZATION ---")
print(df[['Name', 'Salary_USD', 'Centered_Salary_Manual', 'Standardized_Salary_Sklearn']].to_string(index=False))
print("\n" + "="*70 + "\n")

# Verify that the mean is now practically 0
print(f"Verification - Mean of Standardized Salary: {df['Standardized_Salary_Sklearn'].mean():.1f}")